In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tqdm
from source.random_walk import generate_random_walk
from source.plot_tools import activity_map

In [ ]:
n = 128
tau = 10e-3
dt = 0.5e-3

lambda_net = 18
beta = 3.0 / lambda_net**2
gamma = 1.05 * beta
a_weight = 1

l_shift = 2.0
alpha = 0.10315


B0 = 1
seed = 0


def periodic_kernel(n, a_weight, gamma, beta, l_shift=0.0, e_theta=(0.0, 0.0)):
    idx = np.arange(n)
    d = idx - n // 2
    d = np.where(d > n / 2, d - n, d)
    d = np.where(d < -n / 2, d + n, d)
    dx, dy = np.meshgrid(d, d, indexing="ij")

    sx = dx - l_shift * e_theta[0]
    sy = dy - l_shift * e_theta[1]
    r2 = sx**2 + sy**2

    K = a_weight * np.exp(-gamma * r2) - np.exp(-beta * r2)
    K = np.fft.ifftshift(K)
    return K


dir_vectors = {
    "E": np.array([1.0, 0.0]),
    "W": np.array([-1.0, 0.0]),
    "N": np.array([0.0, 1.0]),
    "S": np.array([0.0, -1.0]),
}
dir_pattern = np.array([["E", "N"], ["S", "W"]])
theta_dir = np.tile(dir_pattern, (n // 2, n // 2))

directed_kernels = {}
directed_masks = {}

for preferred_direction in dir_vectors:
    directed_kernels[preferred_direction] = np.fft.fft2(
        periodic_kernel(
            n,
            a_weight,
            gamma,
            beta,
            l_shift=l_shift,
            e_theta=dir_vectors[preferred_direction],
        )
    )
    directed_masks[preferred_direction] = theta_dir == preferred_direction


def recurrent_input(s):
    rec_input = np.zeros_like(s)
    for dir in dir_vectors:
        rec_input += np.real(
            np.fft.ifft2(np.fft.fft2(directed_masks[dir] * s) * directed_kernels[dir])
        )
    return rec_input


e_theta_x = np.vectorize(lambda d: dir_vectors[d][0])(theta_dir)
e_theta_y = np.vectorize(lambda d: dir_vectors[d][1])(theta_dir)


def feedforward_input(vx, vy):
    return B0 * (1.0 + alpha * (e_theta_x * vx + e_theta_y * vy))


def step(s, vx, vy):
    total_input = recurrent_input(s) + feedforward_input(vx, vy)
    return s + (dt / tau) * (-s + np.maximum(total_input, 0.0))

In [ ]:
## Simulation parameters:
n_warmup = 5000
box_size = 1.5
rng = np.random.default_rng(seed=0)

position, velocity, time = generate_random_walk(
    T=500, dt=dt, boxsize=box_size, rng=rng
)


In [ ]:

s = rng.uniform(size=(n, n)) * 0.01
for step_iter in range(n_warmup):
    s = step(s, 0, 0)

n_steps = position.shape[0]
population_records = 1000
record_steps = np.linspace(1, n_steps - 1, population_records, dtype=int)

recorded_population = np.zeros((population_records, n, n))
recorded_sample_neuron = np.zeros(n_steps)

population_recording_index = 0
for step_iter in tqdm.tqdm(range(1, n_steps), "Running steps"):

    vx, vy = velocity[step_iter - 1, 0], velocity[step_iter - 1, 1]
    # fixed schedule for exactly `population_records` full-population snapshots
    population_recording_index = 0

    for step_iter in tqdm.tqdm(range(1, n_steps), "Running steps"):
        vx, vy = velocity[step_iter - 1, 0], velocity[step_iter - 1, 1]
        s = step(s, vx, vy)

        if (
            population_recording_index < recorded_population.shape[0]
            and step_iter == record_steps[population_recording_index]
        ):
            recorded_population[population_recording_index] = s.copy()
            population_recording_index += 1

        recorded_sample_neuron[step_iter] = s[0, 0]

    recorded_sample_neuron[step_iter] = s[0, 0]

In [ ]:
# np.save("simulation_data/recorded_sample_neuron.npy", recorded_sample_neuron)
# np.save("simulation_data/recorded_position.npy", position)
# np.save("simulation_data/recorded_time.npy", time)
# np.save("simulation_data/recorded_population.npy", recorded_population)
# recorded_sample_neuron = np.load("simulation_data/recorded_sample_neuron.npy")
# position = np.load("simulation_data/recorded_position.npy")
# time = np.load("simulation_data/recorded_time.npy")
# recorded_population = np.load("simulation_data/recorded_population.npy")

In [ ]:
binned_activity, edges = activity_map(position, recorded_sample_neuron)
fig, ax = plt.subplots(
    1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": (1, 1.2)}
)
cb = ax[1].imshow(
    binned_activity, origin="lower", extent=[0, box_size, 0, box_size], cmap="jet",interpolation="bilinear",

)
ax[1].set_xlabel("x (m)")
ax[1].set_ylabel("y (m)")
ax[1].set_title("Firing Rate Map")
fig.colorbar(cb, ax=ax[1], label="Average Activity")

ax[0].set_title("Simulated Trajectory")
ax[0].plot(position[:,0], position[:,1], lw=0.3, color="gray")
ax[0].set_xlabel("x (m)")
ax[0].set_ylabel("y (m)")
ax[0].set_xlim((0, box_size))
ax[0].set_ylim((0, box_size))
fig.savefig("plots/can_grid_cells.pdf")

In [ ]:
plt.imshow(recorded_population[-3])